<a href="https://colab.research.google.com/github/ob-s/robust-federated-learning/blob/main/notebooks/colab_runner.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [2]:
# The '!' tells the notebook to run a terminal command instead of Python code
!git clone https://github.com/ob-s/robust-federated-learning.git

# The '%' changes the working directory to your newly downloaded folder
%cd robust-federated-learning

# Install the required libraries we set up earlier
!pip install -r requirements.txt

fatal: destination path 'robust-federated-learning' already exists and is not an empty directory.
/content/robust-federated-learning
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 17.2/17.2 MB 99.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 947.5/947.5 kB 64.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 77.5/77.5 kB 8.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 60.2/60.2 kB 6.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 4.9/4.9 MB 112.5 MB/s eta 0:00:00


In [3]:
!git pull

remote: Enumerating objects: 7, done.
remote: Counting objects: 100% (7/7), done.
remote: Compressing objects: 100% (3/3), done.
remote: Total 4 (delta 2), reused 0 (delta 0), pack-reused 0 (from 0)
Unpacking objects: 100% (4/4), 703 bytes | 703.00 KiB/s, done.
From https://github.com/ob-s/robust-federated-learning
   2e0365e..590e7b5  main       -> origin/main
Updating 2e0365e..590e7b5
Fast-forward
 notebooks/colab_runner.ipynb | 27 ++++++++++++++++-----------
 1 file changed, 16 insertions(+), 11 deletions(-)


In [4]:
import ipywidgets as widgets
from IPython.display import display, clear_output
import time

# Create output console for the evaluators to read
output_console = widgets.Output(layout={'border': '1px solid black', 'height': '200px'})

# Create interactive presentation buttons
btn_clean = widgets.Button(description="1. Run Clean FedAvg", button_style='success')
btn_attack = widgets.Button(description="2. Inject Poison Attack", button_style='danger')
btn_defend = widgets.Button(description="3. Activate Robust Defense", button_style='primary')

def run_clean(b):
    with output_console:
        print("[System] Initiating Standard FedAvg...")
        time.sleep(1)
        print("[Status] Round 1: Accuracy 82% | No anomalies detected.")

def run_attack(b):
    with output_console:
        print("\n[Warning] Malicious Client injected scaled Byzantine vector!")
        time.sleep(1)
        print("[Status] Round 2: Accuracy dropped to 12%. Model corrupted.")

def run_defend(b):
    with output_console:
        print("\n[System] Defense Pipeline Activated (Krum Aggregation).")
        time.sleep(1)
        print("[Security] Outlier detected! Client trust score penalized.")
        print("[Status] Round 3: Accuracy stabilized at 81%. Attack mitigated.")

# Bind functions to button clicks
btn_clean.on_click(run_clean)
btn_attack.on_click(run_attack)
btn_defend.on_click(run_defend)

# Display the control panel
dashboard = widgets.VBox([widgets.HBox([btn_clean, btn_attack, btn_defend]), output_console])
display(dashboard)

In [9]:
!git pull

remote: Enumerating objects: 7, done.
remote: Counting objects: 100% (7/7), done.
remote: Compressing objects: 100% (2/2), done.
remote: Total 4 (delta 2), reused 4 (delta 2), pack-reused 0 (from 0)
Unpacking objects: 100% (4/4), 1.29 KiB | 1.29 MiB/s, done.
From https://github.com/ob-s/robust-federated-learning
   ae812f1..a5f5b1c  main       -> origin/main
Updating ae812f1..a5f5b1c
Fast-forward
 src/server.py | 64 +++++++++++++++++++++++++++++++++++++++++++++++++++++++++++
 1 file changed, 64 insertions(+)


In [10]:
import sys
import torch
import ipywidgets as widgets
from IPython.display import display, clear_output

# Add the src directory to the path so Colab can find your modules
sys.path.append('src')

from data_utils import get_dataset, partition_data
from client import ClientNode, MaliciousClient
from defense import AnomalyDetector, TrustManager
from aggregation import RobustAggregator
from server import AggregationServer

# 1. Environment Setup
device = 'cuda' if torch.cuda.is_available() else 'cpu'
print(f"[System] Initializing on device: {device}")

# Load benchmark dataset and apply non-IID partitioning
trainset, testset = get_dataset('mnist', data_dir='./data')
client_datasets = partition_data(trainset, num_clients=5, iid=False, alpha=0.5)

# 2. UI Dashboard Setup
output_console = widgets.Output(layout={'border': '1px solid black', 'height': '300px', 'padding': '10px'})
btn_clean = widgets.Button(description="1. Run Clean FedAvg", button_style='success')
btn_attack = widgets.Button(description="2. Inject Poison Attack", button_style='danger')
btn_defend = widgets.Button(description="3. Activate Defense", button_style='primary')

# 3. Execution Functions
def run_simulation(scenario):
    with output_console:
        clear_output()
        print(f"--- Starting Scenario: {scenario} ---")

        # Initialize Server
        server = AggregationServer(testset, device=device)

        # Initialize Clients (4 Honest, 1 Malicious)
        clients = []
        for i in range(4):
            clients.append(ClientNode(f"Honest_{i}", client_datasets[i], device=device, epochs=1))

        # The malicious client overrides local SGD to scale weights by 10x
        malicious_client = MaliciousClient("Malicious_4", client_datasets[4], device=device, epochs=1, attack_type='byzantine', scaling_factor=10.0)

        # Configure Aggregator and Defense Pipeline based on the scenario selected[cite: 1]
        if scenario == 'Clean':
            active_clients = clients # No malicious client
            aggregator = RobustAggregator(aggregation_rule='fedavg')
            anomaly_detector, trust_manager = None, None

        elif scenario == 'Attack':
            active_clients = clients + [malicious_client] # Inject attacker
            aggregator = RobustAggregator(aggregation_rule='fedavg') # Vulnerable baseline
            anomaly_detector, trust_manager = None, None

        elif scenario == 'Defense':
            active_clients = clients + [malicious_client]
            # Use Krum aggregation combined with the distance-based anomaly detector[cite: 1]
            aggregator = RobustAggregator(aggregation_rule='krum', num_byzantine=1)
            anomaly_detector = AnomalyDetector(std_threshold=2.0)
            trust_manager = TrustManager(client_ids=[c.client_id for c in active_clients])

        # Run 3 Federated Training Rounds
        for round_num in range(1, 4):
            print(f"\n[Round {round_num}] Training...")
            flags = server.trigger_round(active_clients, aggregator, anomaly_detector, trust_manager)

            # Print security alerts if defense is active
            if flags and any(flags.values()):
                flagged_ids = [cid for cid, flagged in flags.items() if flagged]
                print(f"  [!] SECURITY ALERT: Anomalous updates dropped from {flagged_ids}")

            acc = server.evaluate_global_model()
            print(f"  -> Global Model Accuracy: {acc:.2f}%")

        print("\n--- Simulation Complete ---")

# 4. Bind Functions to Buttons
btn_clean.on_click(lambda b: run_simulation('Clean'))
btn_attack.on_click(lambda b: run_simulation('Attack'))
btn_defend.on_click(lambda b: run_simulation('Defense'))

# Display the interactive control panel
dashboard = widgets.VBox([widgets.HBox([btn_clean, btn_attack, btn_defend]), output_console])
display(dashboard)

[System] Initializing on device: cuda


100%|██████████| 9.91M/9.91M [00:00<00:00, 19.2MB/s]
100%|██████████| 28.9k/28.9k [00:00<00:00, 460kB/s]
100%|██████████| 1.65M/1.65M [00:00<00:00, 4.26MB/s]
100%|██████████| 4.54k/4.54k [00:00<00:00, 7.50MB/s]
